In [ ]:
import json
import requests
import pandas as pd
from datetime import datetime, timezone

def hoje_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")

def make_headers(token: str) -> dict:
    return {
        "Accept": "application/json, text/plain, */*",
        "Authorization": f"Bearer {token}",
        "User-Agent": "Mozilla/5.0 ...",
        "Content-Type": "application/json",
    }

def import_json_payload(path_json: str) -> str:
    with open(path_json, "r", encoding="utf-8") as f:
        json_dict = json.load(f)
    return json.dumps(json_dict, ensure_ascii=False)

def get_codCatalogo_from_payload(path_json: str) -> list[int]:
    with open(path_json, encoding="utf-8") as f:
        json_data = json.load(f)

    solicitacoes_nested = json_data.get("solicitacoes", [])

    solicitacoes_flat = []
    for item in solicitacoes_nested:
        if isinstance(item, list):
            solicitacoes_flat.extend(item)
        else:
            solicitacoes_flat.append(item)

    df = pd.DataFrame(solicitacoes_flat)
    if "codCatalogo" not in df.columns:
        return []
    # remove nulos e dupes preservando ordem
    cods = [int(x) for x in df["codCatalogo"].dropna().tolist()]
    seen = set()
    cods_unique = []
    for c in cods:
        if c not in seen:
            cods_unique.append(c)
            seen.add(c)
    return cods_unique

def fetch_dados_etapa(codigos: list[int], token: str) -> pd.DataFrame:
    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/RelatoriosEtapa/ObterTempoEtapaRelatorio"
    payload = {
        "codCatalogos": codigos,
        "dataInicio": "2022-01-01T00:00:00.000Z",
        "dataFim": hoje_utc_iso(),
        "ativo": 1,
    }
    r = requests.post(url, json=payload, headers=make_headers(token), timeout=60)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data if isinstance(data, list) else data.get("data", []))

def fetch_dados_solicitacoes(payload_str: str, token: str) -> pd.DataFrame:
    url = "https://actogestaoapi-gdhrfgdfc8bbe8hs.brazilsouth-01.azurewebsites.net/api/Tabela/VisualizarDadosIntermediarios"
    config = json.loads(payload_str)

    resp = requests.post(url, headers=make_headers(token), json=config, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    lista_final = []
    if isinstance(data.get("data"), list):
        for item in data["data"]:
            dados_dict = item.get("dados", {})
            if isinstance(dados_dict, dict):
                for _, lista in dados_dict.items():
                    if isinstance(lista, list):
                        lista_final.extend(lista)

    return pd.DataFrame(lista_final)

def extrair_tabela_acto_gestao(path_payload_json: str, token: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    lista_cod_catalogo = get_codCatalogo_from_payload(path_payload_json)

    df_etapas = fetch_dados_etapa(lista_cod_catalogo, token) if lista_cod_catalogo else pd.DataFrame()
    df_solicitacoes = fetch_dados_solicitacoes(import_json_payload(path_payload_json), token)

    print(f"Solicitações: {len(df_solicitacoes)}")
    print(f"OSs tempo/etapa: {df_etapas['seqFluxo'].nunique()}")

    return df_solicitacoes, df_etapas
